# Stage 05 - Governed Batch Scoring

Score the shared systems with model/version lineage and reason fields suitable for analyst review.

In [ ]:
from contextlib import nullcontext
from datetime import datetime, timezone
from pathlib import Path
import importlib.util
import json

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


from pathlib import Path

import numpy as np
import pandas as pd

DATA_FILE = "readiness_observation_features.csv"
FEATURE_TABLE = "silver_readiness_feature_store"
DATA_CANDIDATES = [
    Path("/lakehouse/default/Files") / DATA_FILE,
    Path("../data") / DATA_FILE,
    Path("data") / DATA_FILE,
    Path("Files") / DATA_FILE,
]


def locate_data_file(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    checked = ", ".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"Could not find {DATA_FILE}. Checked: {checked}")


def min_max_scale(series):
    span = series.max() - series.min()
    if span == 0:
        return pd.Series(0.0, index=series.index)
    return (series - series.min()) / span

spark_session = globals().get("spark")
if spark_session is not None and spark_session.catalog.tableExists(FEATURE_TABLE):
    frame = spark_session.table(FEATURE_TABLE).toPandas()
    data_path = f"Lakehouse table {FEATURE_TABLE}"
else:
    data_path = locate_data_file(DATA_CANDIDATES)
    frame = pd.read_csv(data_path)

frame["feature_timestamp_utc"] = pd.to_datetime(frame["feature_timestamp_utc"], utc=True)
frame = frame.sort_values(
    ["scenario_id", "simulation_run_id", "system_instance_id", "feature_timestamp_utc"]
).reset_index(drop=True)
frame["quality_gap"] = 1.0 - frame["quality_rate"]
frame["feature_recency_minutes"] = frame["track_freshness_seconds"] / 60.0
frame["health_decline_flag"] = (frame["health_trend_index"] < 0).astype(int)
frame["maintenance_age_band_index"] = frame["maintenance_age_category"].map(
    {"fresh-service": 0, "steady-cycle": 1, "extended-cycle": 2}
).astype(int)
frame["run_quality_delta"] = frame["quality_rate"] - frame.groupby("simulation_run_id")["quality_rate"].transform("mean")
gap_totals = frame.groupby("simulation_run_id")["gap_count"].transform("sum").replace(0, 1)
frame["run_gap_share"] = frame["gap_count"] / gap_totals
frame["baseline_alignment_gap"] = frame["baseline_deviation_index"] + frame["quality_gap"]
frame["deterministic_baseline_score"] = (
    0.30 * min_max_scale(frame["track_freshness_seconds"])
    + 0.20 * min_max_scale(frame["gap_count"])
    + 0.20 * min_max_scale(frame["quality_gap"])
    + 0.15 * min_max_scale(frame["abstract_ack_lag_seconds"])
    + 0.10 * min_max_scale(frame["baseline_deviation_index"])
    + 0.05 * min_max_scale(frame["maintenance_age_days"])
)

feature_columns = ['track_freshness_seconds', 'gap_count', 'quality_rate', 'abstract_ack_lag_seconds', 'health_trend_index', 'maintenance_age_days', 'maintenance_age_category', 'test_phase', 'baseline_deviation_index', 'late_event_count', 'feature_recency_minutes', 'quality_gap', 'health_decline_flag', 'maintenance_age_band_index', 'run_quality_delta', 'run_gap_share', 'baseline_alignment_gap']
numeric_features = ['track_freshness_seconds', 'gap_count', 'quality_rate', 'abstract_ack_lag_seconds', 'health_trend_index', 'maintenance_age_days', 'baseline_deviation_index', 'late_event_count', 'feature_recency_minutes', 'quality_gap', 'health_decline_flag', 'maintenance_age_band_index', 'run_quality_delta', 'run_gap_share', 'baseline_alignment_gap']
categorical_features = ["maintenance_age_category", "test_phase"]
run_order = frame.groupby("simulation_run_id")["feature_timestamp_utc"].min().sort_values().index.tolist()
selection_train_runs = run_order[:2]
selection_validation_runs = run_order[2:3]
scoring_runs = run_order[:3]
holdout_runs = run_order[3:]
selection_train_frame = frame[frame["simulation_run_id"].isin(selection_train_runs)].copy()
selection_validation_frame = frame[frame["simulation_run_id"].isin(selection_validation_runs)].copy()
scoring_frame = frame[frame["simulation_run_id"].isin(scoring_runs)].copy()
holdout_frame = frame[frame["simulation_run_id"].isin(holdout_runs)].copy()
MLFLOW_AVAILABLE = importlib.util.find_spec("mlflow") is not None


In [ ]:
def prevalence_rank(probabilities, positive_rate):
    top_n = max(1, int(round(len(probabilities) * positive_rate)))
    ranking = pd.Series(probabilities).rank(method="first", ascending=False)
    return (ranking <= top_n).astype(int)


def metric_row(model_name, split_name, y_true, probabilities, positive_rate):
    y_true = pd.Series(y_true).astype(int)
    predicted = prevalence_rank(probabilities, positive_rate)
    return {
        "model_name": model_name,
        "split": split_name,
        "roc_auc": round(float(roc_auc_score(y_true, probabilities)), 4),
        "average_precision": round(float(average_precision_score(y_true, probabilities)), 4),
        "brier_loss": round(float(brier_score_loss(y_true, probabilities)), 4),
        "predicted_priority_rate": round(float(predicted.mean()), 4),
    }


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
            categorical_features,
        ),
    ]
)

candidate_models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=7),
    "random_forest": RandomForestClassifier(n_estimators=250, class_weight="balanced", random_state=7),
}
comparison_rows = []
fitted_models = {}
selection_positive_rate = float(selection_train_frame["synthetic_review_priority_label"].mean())
comparison_rows.append(
    metric_row(
        "deterministic_baseline",
        "selection_validation",
        selection_validation_frame["synthetic_review_priority_label"],
        selection_validation_frame["deterministic_baseline_score"],
        selection_positive_rate,
    )
)
for model_name, estimator in candidate_models.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", estimator)])
    pipeline.fit(selection_train_frame[feature_columns], selection_train_frame["synthetic_review_priority_label"])
    fitted_models[model_name] = pipeline
    probabilities = pipeline.predict_proba(selection_validation_frame[feature_columns])[:, 1]
    comparison_rows.append(
        metric_row(
            model_name,
            "selection_validation",
            selection_validation_frame["synthetic_review_priority_label"],
            probabilities,
            selection_positive_rate,
        )
    )
comparison_frame = pd.DataFrame(comparison_rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
DEMO_SELECTION_MARGIN = 0.02
baseline_auc = comparison_frame[comparison_frame["model_name"] == "deterministic_baseline"].iloc[0]["roc_auc"]
best_candidate = comparison_frame[comparison_frame["model_name"] != "deterministic_baseline"].iloc[0]
selected_model_name = best_candidate["model_name"] if best_candidate["roc_auc"] >= baseline_auc + DEMO_SELECTION_MARGIN else "deterministic_baseline"

mlflow = None
if MLFLOW_AVAILABLE:
    import mlflow

run_context = mlflow.start_run(run_name=f"demo04_batch_scoring_{selected_model_name}", nested=True) if mlflow is not None else nullcontext()
with run_context as active_run:
    if selected_model_name == "deterministic_baseline":
        frame["model_priority_probability"] = frame["deterministic_baseline_score"]
        model_lineage_id = "deterministic-baseline-local"
    else:
        selected_pipeline = Pipeline(
            steps=[("preprocessor", preprocessor), ("model", candidate_models[selected_model_name])]
        )
        selected_pipeline.fit(scoring_frame[feature_columns], scoring_frame["synthetic_review_priority_label"])
        frame["model_priority_probability"] = selected_pipeline.predict_proba(frame[feature_columns])[:, 1]
        model_lineage_id = active_run.info.run_id if active_run is not None else f"local-{selected_model_name}-demo04"
        holdout_probabilities = selected_pipeline.predict_proba(holdout_frame[feature_columns])[:, 1]
        holdout_metrics = metric_row(
            selected_model_name,
            "holdout",
            holdout_frame["synthetic_review_priority_label"],
            holdout_probabilities,
            selection_positive_rate,
        )
    if active_run is not None:
        mlflow.log_params({
            "selected_model_name": selected_model_name,
            "selection_train_runs": ",".join(selection_train_runs),
            "selection_validation_run": selection_validation_runs[0],
            "scoring_runs": ",".join(scoring_runs),
            "holdout_run": holdout_runs[0],
        })
        if selected_model_name != "deterministic_baseline":
            mlflow.log_metric("holdout_roc_auc", holdout_metrics["roc_auc"])
            mlflow.log_metric("holdout_average_precision", holdout_metrics["average_precision"])
            mlflow.log_metric("holdout_brier_loss", holdout_metrics["brier_loss"])

frame["recommended_review_queue"] = np.where(
    prevalence_rank(frame["model_priority_probability"], selection_positive_rate) == 1,
    "priority-review",
    "routine-review",
)
frame["model_name"] = selected_model_name
frame["model_lineage_id"] = model_lineage_id
frame["scored_at_utc"] = datetime.now(timezone.utc).isoformat()
frame["synthetic_use_only"] = "SYNTHETIC / ANALYTICAL DEMO"
frame["human_adjudication_status"] = "pending-analyst-review"

comparison_frame


In [ ]:
scored_columns = [
    "scenario_id",
    "simulation_run_id",
    "test_event_id",
    "site_id",
    "system_instance_id",
    "system_family",
    "feature_timestamp_utc",
    "source_snapshot_id",
    "feature_snapshot_id",
    "baseline_snapshot_id",
    "finding_snapshot_id",
    "model_name",
    "model_lineage_id",
    "quality_rate",
    "track_freshness_seconds",
    "abstract_ack_lag_seconds",
    "health_trend_index",
    "baseline_deviation_index",
    "deterministic_baseline_score",
    "model_priority_probability",
    "recommended_review_queue",
    "synthetic_use_only",
    "human_adjudication_status",
    "scored_at_utc",
]

batch_scoring_summary = {
    "classification": "SYNTHETIC_UNCLASS",
    "selected_model": selected_model_name,
    "model_lineage_id": model_lineage_id,
    "selection_rule": f"Use a learned model only when it exceeds the deterministic baseline by the invented demo margin of {DEMO_SELECTION_MARGIN:.2f} on the validation run.",
    "source_snapshots": sorted(frame["source_snapshot_id"].unique().tolist()),
    "feature_snapshots": sorted(frame["feature_snapshot_id"].unique().tolist()),
    "baseline_snapshots": sorted(frame["baseline_snapshot_id"].unique().tolist()),
    "findings_snapshots": sorted(frame["finding_snapshot_id"].unique().tolist()),
    "limitations": [
        "The ranking is a demonstration aid for analyst review.",
        "It does not set official readiness or requirement status.",
        "MLflow registration is optional and skipped when unavailable.",
    ],
}

print(json.dumps(batch_scoring_summary, indent=2))
spark_session = globals().get("spark")
if spark_session is not None:
    spark_session.createDataFrame(frame[scored_columns]).write.option("overwriteSchema", "true").mode("overwrite").saveAsTable("gold_governed_readiness_scores")
    print("Saved optional Lakehouse table: gold_governed_readiness_scores")
else:
    print("Spark session not detected. Skipping optional Lakehouse table write.")

frame[scored_columns].sort_values(["simulation_run_id", "model_priority_probability"], ascending=[True, False]).reset_index(drop=True)
